# Chapter 5 — Memory

In Chapter 1 memory was a list of messages that vanished when the turn ended. An agent
that forgets everything between incidents cannot notice that the same attacker has now
phished three employees this week.

| Type | What it holds | What it really is |
|---|---|---|
| Working | this turn's messages | a buffer you pay for on every call |
| Episodic | past incidents | a similarity index with a threshold |
| Semantic | durable facts | a knowledge store |
| Procedural | what worked before | a cache with a quality gate |




## Setup

This lab installs from **one** `requirements.txt`.

In [1]:
REPO_URL = "https://github.com/gstripling00/ai-engineer.git"

import os, sys, subprocess

if not os.path.isdir("aegis"):
    result = subprocess.run(["git", "clone", REPO_URL, "aegis"],
                            capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError("git clone failed - check REPO_URL above.\n" + result.stderr)

os.chdir("aegis")
sys.path.insert(0, os.path.abspath("."))
print("repo:", os.getcwd())


repo: /content/aegis


In [2]:
# --no-warn-conflicts silences a cosmetic Colab-only notice about `requests`;
# see the comment block at the top of requirements.txt. Real resolver errors still raise.
!pip -q install --no-warn-conflicts -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.5/247.5 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 695.4/695.4 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23

Now verify the environment before running any lab code. This is the same check CI
runs, and it catches the one dependency conflict that would otherwise waste your
afternoon. It also confirms this chapter's source folder is in the checkout.


In [3]:
!python tools/check_env.py --chapter 5

dependencies
  ok      langgraph              open-source agent track (StateGraph/END)
  ok      langchain-core         message and tool primitives
  ok      langchain-community    RAGAS dependency — see the pin note
  ok      google-adk             Google Cloud agent track (Agent, Workflow)
  ok      mcp                    tool discovery and hardening (Ch 3, 9, 11)
  ok      openai                 the default real-model tier
  ok      langchain-openai       wires OpenAI into RAGAS
  ok      ragas                  evaluation (Ch 10)
  ok      sacrebleu              required by RAGAS BleuScore
  ok      opentelemetry-sdk      tracing (Ch 10)
  ok      chromadb               vector store (Ch 6)
  ok      rank-bm25              sparse retrieval for hybrid search (Ch 6)
  ok      pytest                 the test suite

critical pin
  ok      langchain-community 0.3.29 (compatible with ragas)

model access
  absent  OPENAI_API_KEY not set
          Offline labs still run: AEGIS_MODEL=mock
  

### Choosing a model tier

The labs read `AEGIS_MODEL` and swap the model behind a single seam:

| Tier | Cost | Determinism | Use it for |
|---|---|---|---|
| `mock` | free, no key | identical every run | learning the control flow; the test suite; CI |
| `openai` | billed per call | varies run to run | seeing a real model make these decisions |

Start on `mock`. Everything in this chapter runs there. When you switch to
`openai`, the code does not change — only the seam does.

Set the key from the environment, never as a literal in a cell. In Colab use the
key icon in the sidebar (Secrets); the cell below reads it without printing it.


In [4]:
import os

os.environ["AEGIS_MODEL"] = "mock"     # free, deterministic, no key

# To use a real model instead, uncomment these two lines:
# from getpass import getpass
# os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: "); os.environ["AEGIS_MODEL"] = "openai"

print("model tier:", os.environ["AEGIS_MODEL"])


model tier: mock


## Working memory costs money on every call

Working memory is re-read every turn, so its cost is not paid once — it grows with the
conversation. Three strategies, and they are not interchangeable.

Watch the `keeps system prompt` column. Naive truncation drops from the front, and the
front is where the **constraints** live. An agent that loses its guardrails halfway
through a long investigation has not run out of context — it has been silently
unaligned by a for-loop.


In [5]:
import sys
sys.path.insert(0, "labs/chapter-05-memory")   # this chapter's source lives beside the notebook
from memory.context_budget import count_tokens, truncate, sliding_window, summarize_middle

messages = [{"role": "system",
             "content": "You are Aegis. You may NOT take remediation actions yourself."}]
for i in range(6):
    messages.append({"role": "assistant", "content": f"[call] search_logs(query=auth_fail_{i})"})
    messages.append({"role": "tool", "content": '{"count": 4, "results": [...]}' * 3})

print(f'full memory: {count_tokens(messages)} tokens across {len(messages)} messages')
print()
print(f'{"strategy":18} {"tokens":>7} {"msgs":>5}  keeps system prompt')
for name, fn in (("truncate", truncate),
                 ("sliding_window", sliding_window),
                 ("summarize_middle", summarize_middle)):
    kept = fn(messages, 60)
    has_system = any(m["role"] == "system" and "may NOT take remediation" in m["content"]
                     for m in kept)
    print(f'{name:18} {count_tokens(kept):7} {len(kept):5}  {has_system}')
print()
print("truncate() lost the guardrail. That is the bug, and it is silent.")


full memory: 240 tokens across 13 messages

strategy            tokens  msgs  keeps system prompt
truncate                37     2  False
sliding_window          55     3  True
summarize_middle        36     2  True

truncate() lost the guardrail. That is the bug, and it is silent.


## Episodic memory: three reports become one campaign

Nothing about the third report is special. The **memory** is what makes it different —
and note that no rule anywhere says "three reports is a campaign." The agent recalls
similar past incidents and counts.


In [6]:
from memory.memory_store import EpisodicMemory, ProceduralMemory, assess_with_memory

BAD = "helpdesk@it-support-reset.example"
PLAYBOOK = ["quarantine message", "reset credentials", "notify user"]

episodic, procedural = EpisodicMemory(), ProceduralMemory()

for i, user in enumerate(["j.okafor", "m.chen", "a.singh"]):
    report = {"id": f"INC-{i}", "category": "phishing", "sender": BAD, "user": user}
    assessment = assess_with_memory(report, episodic, procedural)   # RECALL first
    episodic.record(report)                                          # then record
    procedural.learn("phishing", PLAYBOOK, succeeded=True)

    label = "CAMPAIGN" if assessment["is_campaign"] else "isolated"
    print(f'{report["id"]}: {label:8} severity={assessment["recommended_severity"]:6} '
          f'(recalled {len(assessment["related_prior"])} prior)')


INC-0: isolated severity=medium (recalled 0 prior)
INC-1: isolated severity=medium (recalled 1 prior)
INC-2: CAMPAIGN severity=high   (recalled 2 prior)


## Semantic memory

Not a record of what happened — a record of what is **true**. Known-bad senders, asset
owners, which accounts are privileged.


In [7]:
from memory.memory_store import SEMANTIC_FACTS, is_known_bad_sender

print("known bad sender:", is_known_bad_sender(BAD))
print("unknown sender:  ", is_known_bad_sender("newsletter@marketing.example"))
print()
print("fact categories held:", list(SEMANTIC_FACTS))


known bad sender: True
unknown sender:   False

fact categories held: ['known_bad_senders', 'asset_owners', 'privileged_accounts']


## Procedural memory: only successes teach

Episodic memory remembers *what* happened; procedural remembers *how* it was handled
successfully. Two rules make it a cache with a quality gate rather than a log: failures
do not teach, and a proven playbook is not displaced by a weaker rival.

Without those rules you have built a system that faithfully memorizes its own mistakes.


In [8]:
proc = ProceduralMemory()

proc.learn("phishing", PLAYBOOK, succeeded=False)
print("after a FAILED run:  ", proc.recall("phishing"))

proc.learn("phishing", PLAYBOOK, succeeded=True)
proc.learn("phishing", PLAYBOOK, succeeded=True)
print("after two successes: ", proc.recall("phishing"))

proc.learn("phishing", ["do nothing"], succeeded=True)
print("after a weak rival:  ", proc.recall("phishing"))
print()
print("The proven playbook survived. Reuse instead of rediscovery - and")
print("rediscovery costs tokens on every similar incident.")


after a FAILED run:   None
after two successes:  {'steps': ['quarantine message', 'reset credentials', 'notify user'], 'successes': 2}
after a weak rival:   {'steps': ['quarantine message', 'reset credentials', 'notify user'], 'successes': 2}

The proven playbook survived. Reuse instead of rediscovery - and
rediscovery costs tokens on every similar incident.


## The threshold is a measured decision, not a default

`recall()` takes a threshold. Everything above it is "similar"; everything below is
invisible. That single number decides what the agent believes.

Add an unrelated marketing newsletter to memory and sweep it.


In [9]:
mem = EpisodicMemory()
for i in range(3):
    mem.record({"id": f"P{i}", "category": "phishing", "sender": BAD})
mem.record({"id": "N1", "category": "newsletter", "sender": "newsletter@marketing.example"})

query = f"phishing {BAD}"
print(f'{"threshold":>10} {"recalled":>9}   which')
for t in (0.05, 0.10, 0.20, 0.40, 0.60, 0.80):
    hits = mem.recall(query, k=10, threshold=t)
    print(f'{t:>10} {len(hits):>9}   {[h["id"] for h in hits]}')
print()
print("Too low and the newsletter joins the campaign - the SOC investigates a mailing list.")
print("Too high and the real campaign disappears.")
print("There is a right answer and it is a MEASUREMENT, not a default (Ch 10 builds it).")


 threshold  recalled   which
      0.05         4   ['P0', 'P1', 'P2', 'N1']
       0.1         4   ['P0', 'P1', 'P2', 'N1']
       0.2         3   ['P0', 'P1', 'P2']
       0.4         3   ['P0', 'P1', 'P2']
       0.6         3   ['P0', 'P1', 'P2']
       0.8         0   []

Too low and the newsletter joins the campaign - the SOC investigates a mailing list.
Too high and the real campaign disappears.
There is a right answer and it is a MEASUREMENT, not a default (Ch 10 builds it).


---

## What you built

Four memory types, an agent that turns three isolated reports into one recognized
campaign, and two dials that silently set quality: the context budget and the
similarity threshold.

- **Recall before you record.** Reverse it and every incident matches itself.
- **Truncate from the front and you drop your guardrails.**
- **An untuned threshold is a random number generator with good manners.**

**Next:** Chapter 6 grounds Aegis in your runbooks — and shows that how you chunk them
is the biggest lever on whether retrieval finds the right one.
